# 02 — Feature Engineering
## SG Decomposition, Time Weighting, Course Features

**Purpose:** Transform raw round-level SG data into model-ready features.

**Pipeline:**
1. **Strokes Gained Processing** — Normalize, decompose into OTT/APP/ARG/PUTT
2. **Field Strength Adjustment** — Adjust SG for opponent quality
3. **Time Weighting (EWMA)** — Recent rounds count more than old rounds
4. **Course Features** — Build γ_c characteristic vectors
5. **Master Pipeline** — Chain everything into one call


In [ ]:
# --- Setup ---
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config.settings import Settings
from data.loader import DataLoader
from features.strokes_gained import StrokesGainedProcessor, FieldStrengthCalculator
from features.time_weighting import TimeWeighter
from features.course_features import CourseFeatureProcessor
from features.pipeline import FeaturePipeline

cfg = Settings()
loader = DataLoader(cfg)
rounds_df = loader.load_rounds()
print(f"Loaded {len(rounds_df):,} rounds")


In [ ]:
from features.pipeline import FeaturePipeline

rounds_df["date"] = pd.to_datetime(rounds_df["year"].astype(str) + "-01-01")
rounds_df = rounds_df.rename(columns={"dg_id": "player_id"})

pipeline = FeaturePipeline(cfg)
features = pipeline.run(rounds_df)

# Access individual outputs:
features.player_features    # EWMA SG estimates per player
features.field_strength     # field strength per event
features.rounds_enriched    # rounds with field-strength adjustments
features.metadata           # pipeline run info


In [ ]:
# --- Step 2: Field Strength Adjustment ---
fs_calc = FieldStrengthCalculator(cfg)

# Compute field strength per event
field_strengths = fs_calc.compute_field_strength(rounds_df)  # ← fix hereprint(f"\nField strength by event:")
if isinstance(field_strengths, pd.DataFrame):
    print(field_strengths.describe().round(3))
elif isinstance(field_strengths, dict):
    vals = list(field_strengths.values())
    print(f"  Mean: {np.mean(vals):.3f}, Std: {np.std(vals):.3f}")
    print(f"  Range: {np.min(vals):.3f} to {np.max(vals):.3f}")


In [ ]:
# --- Step 3: Time Weighting (EWMA) ---
time_weighter = TimeWeighter(cfg)

# Pick a well-known player to inspect
# (Replace with an actual player_id from your data)
sample_player = rounds_df["player_id"].value_counts().index[0]
player_rounds = rounds_df[rounds_df["player_id"] == sample_player].copy()

print(f"Sample player ID: {sample_player}")
print(f"Total rounds: {len(player_rounds)}")

# Compute time-weighted SG
weighted = time_weighter.compute_weighted_sg(player_rounds)
print(f"\nTime-weighted SG estimate: {weighted}")

# Visualize weight decay
from utils.helpers import exponential_weights
n = min(100, len(player_rounds))
weights = exponential_weights(n, half_life=cfg.EWMA_HALF_LIFE_ROUNDS)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(n), weights, color="#2563eb", alpha=0.7)
ax.set_xlabel(f"Rounds ago (0 = most recent)")
ax.set_ylabel("Weight")
ax.set_title(f"EWMA Weights (half-life = {cfg.EWMA_HALF_LIFE_ROUNDS} rounds)")
ax.axvline(x=cfg.EWMA_HALF_LIFE_ROUNDS, color="red", linestyle="--",
           label=f"Half-life ({cfg.EWMA_HALF_LIFE_ROUNDS})")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# --- Step 4: Course Features ---
# Check if course feature data is available
try:
    course_df = loader.load_course_features()
    course_processor = CourseFeatureProcessor(cfg)
    course_features = course_processor.process(course_df)
    print(f"Course features: {course_features.shape}")
    course_features.head()
except FileNotFoundError:
    print("No course_features.csv found.")
    print("Course features are optional for Phase 1-3. Needed for Phase 4 (course-fit).")
    print("You can create this file manually or pull from DataGolf later.")


In [ ]:
# --- Step 5: Full Feature Pipeline ---
pipeline = FeaturePipeline(cfg)

# Run the complete pipeline
features = pipeline.run(rounds_df)
print(f"\nFeature pipeline output type: {type(features)}")
print(f"Feature pipeline output: {features}")


---
## ✅ Feature Engineering Complete

**Key outputs:**
- Time-weighted SG estimates per player per component
- Field-strength-adjusted metrics
- Course feature vectors (if data available)

**Next step:** → **03_baseline_model.ipynb** (fit the "beat this" baseline)
